# Qwen2.5-Math-1.5B Evaluation with MoTTT
This notebook evaluates the baseline `Qwen/Qwen2.5-Math-1.5B-Instruct` and compares it to `Qwen/Qwen2.5-Math-1.5B` enhanced with the MoTTT (Test-Time LoRA Scratchpad) method.

In [ ]:
!git clone https://github.com/Jerryliu3547/MoTTT.git /content/MoTTT
%cd /content/MoTTT
!pip install -q -e .


In [ ]:
import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from mottt.models.mottt_model import MoTTTModel


## 1. Evaluate Baseline (Qwen2.5-Math-1.5B-Instruct)

In [ ]:
baseline_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading {baseline_name}...")
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_name, trust_remote_code=True)
baseline_model = AutoModelForCausalLM.from_pretrained(
    baseline_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

baseline_pipe = pipeline(
    "text-generation",
    model=baseline_model,
    tokenizer=baseline_tokenizer,
    max_new_tokens=512,
    do_sample=False
)


In [ ]:
import json
import re
from tqdm import tqdm

def extract_predicted_answer(text: str) -> str:
    if "####" in text:
        after_hash = text.split("####")[-1].strip()
        cleaned = re.sub(r"[,\$]", "", after_hash).strip()
        tokens = cleaned.split()
        if tokens:
            return tokens[0].strip()
    match = re.search(r"(?:the\s+answer\s+is\s+|is\s+|equal\s+to\s+)([-+]?\d+(?:\.\d+)?)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "").strip()
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return ""

# Make sure to upload gsm8k_distractor_all.jsonl to your Colab /content/ folder first!
test_file = "/content/gsm8k_distractor_all.jsonl"
if not os.path.exists(test_file):
    print(f"Warning: {test_file} not found. Please upload your test dataset to Colab.")
else:
    records = []
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
            if len(records) >= 100:
                break
    
    print(f"Loaded {len(records)} test records.")
    correct = 0
    
    for rec in tqdm(records, desc="Evaluating Baseline"):
        ctx = rec.get('distractor_context', '')[:1500]
        query = rec.get('query', '')
        gold = str(rec.get('gold_answer', '')).strip()
        
        prompt = f"Background Context:\n{ctx}\n\nQuestion:\n{query}\n\nPlease solve the problem step by step and end your response with '#### [final numerical answer]'."
        outputs = baseline_pipe(prompt)
        gen_text = outputs[0]["generated_text"][len(prompt):].strip()
        pred = extract_predicted_answer(gen_text)
        
        if pred == gold:
            correct += 1
            
    print(f"\nBaseline Accuracy on First {len(records)} Samples: {correct/len(records)*100:.1f}% ({correct}/{len(records)})")


## 2. Evaluate MoTTT Model (Qwen2.5-Math-1.5B + Query-Aware Router)

In [ ]:
# Note: In a real scenario, you'd load Qwen/Qwen2.5-Math-1.5B base model, 
# and load trained mottt_router.pt and reasoning_experts.pt from your checkpoint directory.

base_name = "Qwen/Qwen2.5-Math-1.5B"
print(f"Loading {base_name} for MoTTT...")
mottt_tokenizer = AutoTokenizer.from_pretrained(base_name, trust_remote_code=True)
hf_backbone = AutoModelForCausalLM.from_pretrained(
    base_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

mottt_model = MoTTTModel(
    hidden_dim=hf_backbone.config.hidden_size,
    num_reasoning_experts=4, # Assuming default config
    rank=16,
    alpha=16.0,
    base_backbone=hf_backbone,
    all_linear=True
).to(device)

# Example: Load your trained checkpoints here
# ckpt_dir = Path("../gsm8k/checkpoints")
# mottt_model.router.load_state_dict(torch.load(ckpt_dir / "mottt_router.pt"))
# Add loading logic for reasoning_experts.pt here


In [ ]:
# Create pipeline with adapted backbone
mottt_pipe = pipeline(
    "text-generation",
    model=hf_backbone,
    tokenizer=mottt_tokenizer,
    max_new_tokens=512,
    do_sample=False
)

if 'records' in locals() and len(records) > 0:
    correct = 0
    inner_steps = 1
    inner_opt = torch.optim.SGD(mottt_model.get_scratchpad_parameters(), lr=1e-3)
    
    for rec in tqdm(records, desc="Evaluating MoTTT"):
        ctx = rec.get('distractor_context', '')[:1500]
        query = rec.get('query', '')
        gold = str(rec.get('gold_answer', '')).strip()
        
        # 1. Inner Loop Adaptation
        mottt_model.reset_scratchpad()
        mottt_model.set_routing_gates(None)
        
        ctx_enc = mottt_tokenizer(ctx, truncation=True, max_length=256, return_tensors="pt").to(device)
        for _ in range(inner_steps):
            ctx_out = hf_backbone(input_ids=ctx_enc.input_ids, attention_mask=ctx_enc.attention_mask)
            shift_logits = ctx_out.logits[..., :-1, :].contiguous()
            shift_labels = ctx_enc.input_ids[..., 1:].contiguous()
            inner_loss = F.cross_entropy(shift_logits.view(-1, hf_backbone.config.vocab_size), shift_labels.view(-1))
            inner_opt.zero_grad()
            inner_loss.backward()
            inner_opt.step()
            
        # 2. Query-Aware Routing
        q_enc = mottt_tokenizer(query, return_tensors="pt").to(device)
        with torch.no_grad():
            q_emb = hf_backbone.model.embed_tokens(q_enc.input_ids).mean(dim=1)
            gates, _ = mottt_model.router(q_emb.unsqueeze(0).unsqueeze(0), q_emb.unsqueeze(0))
        mottt_model.set_routing_gates(gates)
        
        # 3. Generation
        prompt = f"Background Context:\n{ctx}\n\nQuestion:\n{query}\n\nPlease solve the problem step by step and end your response with '#### [final numerical answer]'."
        outputs = mottt_pipe(prompt)
        gen_text = outputs[0]["generated_text"][len(prompt):].strip()
        pred = extract_predicted_answer(gen_text)
        
        if pred == gold:
            correct += 1
            
        mottt_model.reset_scratchpad()
        
    print(f"\nMoTTT Accuracy on First {len(records)} Samples: {correct/len(records)*100:.1f}% ({correct}/{len(records)})")
else:
    print("Please load the test records first.")
